In [1]:
import pandas as pd
import random
import numpy as np
import torch
def set_seed(seed):
    """Set seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [2]:
dfCovid = pd.read_csv('data/dataRwaCovid.csv')

In [3]:
dfDiabetes = pd.read_csv('data/dataIHADiabetes.csv')

In [4]:
DiabetesTopics = [
    "Physical",
    "Psychological",
    "No Symptoms",
    "Prognosis",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Testing/Monitoring Devices",
    "Health Data",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Adverse Events",
    "Therapeutic Devices",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Insurance/Billing",
    "Medical Records",
    "Referrals",
    "Transportation",
    "Primary (Pharmaceutical Prevention)",
    "Primary (Non-Pharmaceutical Prevention)",
    "Secondary (Pharmaceutical Prevention)",
    "Secondary (Non-Pharmaceutical Prevention)",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Entertainment",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety Concerns",
    "Health Education",
    "Sexual & Reproductive Health",
    "Child & Family Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "Rapport",
    "Transition to Adult Clinic"
]

In [5]:
CovidTopics = [
    "Physical",
    "Mental/Emotional",
    "No Symptoms",
    "Laboratory/Testing",
    "Imaging",
    "Clinical",
    "Diagnostic Methods - Other",
    "Medications",
    "Procedures",
    "Alternative",
    "Physical Therapy",
    "Counseling",
    "Treatment(Rx) - Other",
    "Outpatient Logistics/Scheduling",
    "Hospitalizations",
    "Pharmaceutical Prevention",
    "Non-Pharmaceutical Prevention",
    "Diet/Nutrition",
    "Exercise",
    "Substance Use",
    "Lifestyle - Other",
    "Housing",
    "Work/School",
    "Social Services",
    "Friends & Family",
    "Cultural/Religion",
    "Travel",
    "Physical Environment/Climate",
    "Financial",
    "Social - Other",
    "Technical/IT",
    "Safety concern",
    "Health Education",
    "Maternal & Child Health",
    "Problems Solved",
    "Grateful Patient",
    "Service Complaint",
    "Request to Stop",
    "Emergent",
    "Urgent",
    "Non-urgent",
    "Stigma Present",
    "wave",
    "batch"
]

In [6]:
commonSubtopics = sorted(list(set(DiabetesTopics) & set(CovidTopics)))

In [7]:
dfCovid[commonSubtopics].sum()

,0
Alternative,35
Clinical,5
Counseling,0
Cultural/Religion,154
Diagnostic Methods - Other,5
Diet/Nutrition,102
Emergent,138
Exercise,25
Financial,53
Friends & Family,138


In [8]:
diabetesColumns = commonSubtopics[:]
diabetesColumns.insert(0,"conversation")
dfDiabetesNew = dfDiabetes[diabetesColumns]


In [9]:
covidColumns = commonSubtopics[:]
covidColumns.insert(0,"conversation(english_only)")
dfCovidNew = dfCovid[covidColumns]


In [10]:
dfCovidNew = dfCovidNew.rename(columns={"conversation(english_only)":"conversation"})

df

In [11]:
dfCombined = pd.concat([dfCovidNew, dfDiabetesNew])

In [12]:
dfCombined.sum()

,0
conversation,"(S): Hello, how are you today? Do you have any..."
Alternative,35
Clinical,6
Counseling,0
Cultural/Religion,343
Diagnostic Methods - Other,8
Diet/Nutrition,340
Emergent,140
Exercise,131
Financial,53


In [13]:
n = 100
label_sum = dfCombined[commonSubtopics].sum()
label_keep = label_sum[label_sum >= n].index.tolist()
label_keep_original = label_keep[:]
label_keep.insert(0,"conversation")

dfCombined_filtered = dfCombined[label_keep]
dfCombined_filtered.shape


(3907, 20)

In [14]:
dfCombined_filtered.sum()

,0
conversation,"(S): Hello, how are you today? Do you have any..."
Cultural/Religion,343
Diet/Nutrition,340
Emergent,140
Exercise,131
Friends & Family,291
Grateful Patient,291
Health Education,322
Laboratory/Testing,1078
Medications,418


In [15]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].str.lower()

/tmp/ipykernel_394/1116881674.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].str.lower()


In [16]:
import nltk
import re
import string

In [17]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.6 MB/s eta 0:00:00


In [18]:
import contractions

In [19]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))

/tmp/ipykernel_394/2646981620.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([contractions.fix(word) for word in x.split()]))


In [20]:
dfCombined_filtered.shape

(3907, 20)

In [21]:
dfCombined_filtered.head()

,conversation,Cultural/Religion,Diet/Nutrition,Emergent,Exercise,Friends & Family,Grateful Patient,Health Education,Laboratory/Testing,Medications,No Symptoms,Non-urgent,Outpatient Logistics/Scheduling,Physical,Physical Environment/Climate,Service Complaint,Social Services,Technical/IT,Urgent,Work/School
0,"(s): hello, how are you today? do you have any...",1,0,0,0,0,1,0,1,0,1,1,0,0,0,0,0,0,0,0
1,(s): welcome to rwanda ministry of health plat...,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0
2,(s): welcome to rwanda ministry of health plat...,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0
3,"(s): hello, how are you today? do you have any...",0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0
4,(s): welcome to rwanda ministry of health plat...,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0


In [22]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'\d+','',x))

/tmp/ipykernel_394/3892837535.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'\d+','',x))


In [23]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'[.]',' ',x))

/tmp/ipykernel_394/1879545715.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: re.sub(r'[.]',' ',x))


In [24]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: "".join([char for char in x if char not in string.punctuation]))

/tmp/ipykernel_394/2885321218.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: "".join([char for char in x if char not in string.punctuation]))


In [25]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [26]:
from nltk.corpus import stopwords

In [27]:
sw = stopwords.words('english')

In [28]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([word for word in x.split() if word not in sw]))

/tmp/ipykernel_394/280526503.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lambda x: " ".join([word for word in x.split() if word not in sw]))


In [29]:
import nltk
nltk.download("punkt_tab")
nltk.download('punkt')
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download('averaged_perceptron_tagger')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

from nltk.corpus import wordnet

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_with_pos(text):
    tokens = nltk.word_tokenize(text)
    tagged_tokens = nltk.pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in tagged_tokens])



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [30]:
dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)

/tmp/ipykernel_394/306948834.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered["conversation"] = dfCombined_filtered["conversation"].apply(lemmatize_with_pos)


In [31]:
dfCombined_filtered.head()
dfCombined_filtered.shape

(3907, 20)

In [32]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

dfCombined_filtered['conversation_tokens'] = dfCombined_filtered['conversation'].apply(word_tokenize)



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/tmp/ipykernel_394/4092647881.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfCombined_filtered['conversation_tokens'] = dfCombined_filtered['conversation'].apply(word_tokenize)


In [33]:
!pip install scikit-multilearn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 5.5 MB/s eta 0:00:00


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
from skmultilearn.model_selection import iterative_train_test_split
from sklearn.model_selection import train_test_split


In [35]:
tf = TfidfVectorizer(max_features=5000,ngram_range=(1,2),min_df=3,max_df=0.6 )
x =dfCombined_filtered["conversation"]

In [36]:
import torch
# from sklearn.multi
# torch.cuda.get_device_name()

In [37]:
y = dfCombined_filtered[label_keep_original].to_numpy()
print(y.shape)

(3907, 19)


In [38]:

import numpy as np

split = np.load("data/shared_split_indices.npz")
train_idx = split["train_idx"]
val_idx   = split["val_idx"]
test_idx  = split["test_idx"]

x_raw = dfCombined_filtered["conversation"].to_numpy()
y     = dfCombined_filtered[label_keep_original].to_numpy()

x_train_raw = x_raw[train_idx]
x_val_raw   = x_raw[val_idx]
x_test_raw  = x_raw[test_idx]

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

df_aug_train = pd.read_csv("data/backtranslated_train.csv")

import pandas as pd
import numpy as np
import re
import string
import contractions
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

df_aug_train = pd.read_csv('data/backtranslated_train.csv')

y_train = df_aug_train[label_keep_original].to_numpy()

sw = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def lemmatize_with_pos(text):
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return " ".join([lemmatizer.lemmatize(tok, get_wordnet_pos(pos)) for tok, pos in tagged])

def preprocess(text):
    text = text.lower()
    text = " ".join([contractions.fix(word) for word in text.split()])
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[.]', ' ', text)
    text = "".join([c for c in text if c not in string.punctuation])
    text = " ".join([w for w in text.split() if w not in sw])
    text = lemmatize_with_pos(text)
    return text

df_aug_train['conversation'] = df_aug_train['conversation'].apply(preprocess)
x_train = df_aug_train["conversation"].to_numpy()

y_train = df_aug_train[label_keep_original].to_numpy()

x_train = tf.fit_transform(pd.Series(x_train.flatten()))
x_val   = tf.transform(pd.Series(x_val_raw.flatten()))
x_test  = tf.transform(pd.Series(x_test_raw.flatten()))

In [39]:
x_train.shape

(5468, 5000)

In [40]:
x_train.shape
type(x_train)


scipy.sparse._csr.csr_matrix

In [41]:
y_train.shape
type(y_train)

numpy.ndarray

In [42]:
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier


In [43]:
# x_val,  y_val,  x_test,y_test = iterative_train_test_split(x_, y_, test_size=0.5)


In [44]:
!pip install imbalanced-learn

In [45]:
!pip install --upgrade scikit-multilearn


In [46]:
y_val.shape

(589, 19)

In [47]:
!pip install xgboost

In [48]:
from xgboost import XGBClassifier
import xgboost as xgb
from sklearn.multiclass import OneVsRestClassifier

In [49]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 12.4 MB/s eta 0:00:00


In [50]:
import optuna

In [51]:
dmatrix_train = xgb.DMatrix(x_train)
dmatrix_val = xgb.DMatrix(x_val)
dmatrix_test = xgb.DMatrix(x_test)


In [52]:
from xgboost import XGBClassifier
from sklearn.multiclass import OneVsRestClassifier
import numpy as np
from sklearn.metrics import f1_score
print(y_train.shape)
print(type(y_train))
print(type(y_val))
print(type(y_test))

(5468, 19)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [53]:
models = {}
list_params = {}

tempDmatrix_val = xgb.DMatrix(x_val)
tempDmatrix_test = xgb.DMatrix(x_test)
for i,label_name in enumerate(label_keep_original):
    print(label_name)
    dmatrix_train.set_label(y_train[:,i])
    dmatrix_val.set_label(y_val[:,i])
    dmatrix_test.set_label(y_test[:,i])
    def objective(trial):
      params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'booster': 'gbtree',
        'subsample': trial.suggest_float('subsample', 0.3, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree',0.3,1.0),
        'scale_pos_weight':(len(y_train[:,i])-y_train[:,i].sum())/y_train[:,i].sum(),

        'eta': trial.suggest_float('eta', 0.01, 1.5),
        'max_depth': trial.suggest_int('max_depth', 1, 10),
        'seed': 42,
        'n_jobs': -1,
        'reg_lambda': trial.suggest_int('reg_lambda',0.5,4),
        'reg_alpha':trial.suggest_int('reg_alpha',0.5,4),
        'gamma': trial.suggest_float('gamma', 0.1, 1),
        'device': 'cuda',
        'use_label_encoder': False,
      }
      eval_list = [(dmatrix_val, 'validation')]
      model = xgb.train(params=params, dtrain=dmatrix_train,evals=eval_list,num_boost_round=1000,verbose_eval=False,early_stopping_rounds=30)


      x_val_predic = model.predict(tempDmatrix_val)


      best_f1 = 0
      best_thresh = 0.5
      for threshold_val in np.arange(0.05, 0.96, 0.05):
          preds = (x_val_predic > threshold_val).astype(int)
          # print(preds.shape)
          # print(y_val[:, i].shape)
          current_f1 = f1_score(y_val[:, i], preds)
          if current_f1 > best_f1:
              best_f1 = current_f1
              best_thresh = threshold_val
      return best_f1
    study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
    study.optimize(objective, n_trials=50)
    print(f"Best trial for {label_name}: F1 score= {study.best_value:.4f}")
    print(f"Best hyperparameters: {study.best_params}")
    list_params[label_name] = study.best_params

[I 2026-05-25 15:37:31,906] A new study created in memory with name: no-name-d88f246c-29df-4266-bab4-4ae28a6752d5


Cultural/Religion


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:37:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2026-05-25 15:37:32,583] Trial 0 finished with value: 0.9504950495049505 and parameters: {'subsample': 0.945404102593965, 'colsample_bytree': 0.7397939550373951, 'eta': 1.1156456305775933, 'max_depth': 1, 'reg_lambda': 0, 'reg_alpha': 2, 'gamma': 0.5102950721156049}. Best is trial 0 with value: 0.9504950495049505.
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:37:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2026-05-25 15:37:33,809] Trial 1 finished with value: 0.9292929292929293 and parameters: {'subsample': 0.3821247181031435, 'colsample_bytree': 0.7080305563987621, 'eta': 0.23981479297932812, 'ma

Best trial for Cultural/Religion: F1 score= 0.9608
Best hyperparameters: {'subsample': 0.8053909240762616, 'colsample_bytree': 0.7707822841814779, 'eta': 1.042346194360421, 'max_depth': 1, 'reg_lambda': 0, 'reg_alpha': 2, 'gamma': 0.3904913654386146}
Diet/Nutrition


[I 2026-05-25 15:38:05,548] Trial 0 finished with value: 0.8118811881188119 and parameters: {'subsample': 0.335545306083955, 'colsample_bytree': 0.6949397346881581, 'eta': 0.34055581597635126, 'max_depth': 7, 'reg_lambda': 1, 'reg_alpha': 2, 'gamma': 0.6818189551675108}. Best is trial 0 with value: 0.8118811881188119.
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:38:05] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2026-05-25 15:38:06,690] Trial 1 finished with value: 0.8932038834951457 and parameters: {'subsample': 0.6645000256152709, 'colsample_bytree': 0.6772864580754918, 'eta': 0.22589633113517452, 'max_depth': 7, 'reg_lambda': 1, 'reg_alpha': 3, 'gamma': 0.825418455891992}. Best is trial 1 with value: 0.8932038834951457.
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:38:06] WARNING: /__w/xgboost/xgboost

...

Best trial for Technical/IT: F1 score= 0.7368
Best hyperparameters: {'subsample': 0.9956986071698033, 'colsample_bytree': 0.9452188375939842, 'eta': 0.5474821965991338, 'max_depth': 8, 'reg_lambda': 2, 'reg_alpha': 2, 'gamma': 0.7171872204130991}
Urgent


[I 2026-05-25 15:50:40,281] Trial 0 finished with value: 0.38961038961038963 and parameters: {'subsample': 0.6251034009023412, 'colsample_bytree': 0.6162119154005151, 'eta': 0.1219460323478835, 'max_depth': 3, 'reg_lambda': 2, 'reg_alpha': 4, 'gamma': 0.8874728876248497}. Best is trial 0 with value: 0.38961038961038963.
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:50:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2026-05-25 15:50:41,248] Trial 1 finished with value: 0.35294117647058826 and parameters: {'subsample': 0.8275677978842939, 'colsample_bytree': 0.4926997945505179, 'eta': 0.2144222345954035, 'max_depth': 6, 'reg_lambda': 1, 'reg_alpha': 3, 'gamma': 0.8743795193042657}. Best is trial 0 with value: 0.38961038961038963.
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:50:41] WARNING: /__w/xgboost/xgb

Best trial for Urgent: F1 score= 0.4299
Best hyperparameters: {'subsample': 0.9897243851797022, 'colsample_bytree': 0.3672481574890575, 'eta': 0.015688729028376175, 'max_depth': 4, 'reg_lambda': 1, 'reg_alpha': 3, 'gamma': 0.9975442229103662}
Work/School


[I 2026-05-25 15:51:47,294] Trial 0 finished with value: 0.7755102040816326 and parameters: {'subsample': 0.47295499185484635, 'colsample_bytree': 0.6785295275232998, 'eta': 0.10852283980275362, 'max_depth': 8, 'reg_lambda': 0, 'reg_alpha': 2, 'gamma': 0.32275494071042266}. Best is trial 0 with value: 0.7755102040816326.
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:51:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2026-05-25 15:51:47,441] Trial 1 finished with value: 0.3191489361702128 and parameters: {'subsample': 0.7142240613170943, 'colsample_bytree': 0.7073497461520213, 'eta': 1.2487744693426017, 'max_depth': 3, 'reg_lambda': 0, 'reg_alpha': 2, 'gamma': 0.6659471179584987}. Best is trial 0 with value: 0.7755102040816326.
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [15:51:47] WARNING: /__w/xgboost/xgbo

Best trial for Work/School: F1 score= 0.8372
Best hyperparameters: {'subsample': 0.5685888706475029, 'colsample_bytree': 0.808749336565037, 'eta': 0.20720242066677816, 'max_depth': 6, 'reg_lambda': 0, 'reg_alpha': 0, 'gamma': 0.2845551201361284}


In [54]:
models = {}
for i,label_name in enumerate(label_keep_original):

  dmatrix_train.set_label(y_train[:,i])
  dmatrix_val.set_label(y_val[:,i])
  dmatrix_test.set_label(y_test[:,i])



  eval_list = [(dmatrix_val, 'validation')]
  model = xgb.train(params=list_params[label_name], dtrain=dmatrix_train,evals=eval_list,num_boost_round=1500,verbose_eval=False,early_stopping_rounds=30)
  models[label_name] = model




In [55]:
models

{'Cultural/Religion': <xgboost.core.Booster at 0x780a134af260>,
 'Diet/Nutrition': <xgboost.core.Booster at 0x780a13482300>,
 'Emergent': <xgboost.core.Booster at 0x780a13482120>,
 'Exercise': <xgboost.core.Booster at 0x780a11a772f0>,
 'Friends & Family': <xgboost.core.Booster at 0x780a0ff91f70>,
 'Grateful Patient': <xgboost.core.Booster at 0x780a0ff92480>,
 'Health Education': <xgboost.core.Booster at 0x780a0ff93890>,
 'Laboratory/Testing': <xgboost.core.Booster at 0x780a0ff907a0>,
 'Medications': <xgboost.core.Booster at 0x780a0ff920c0>,
 'No Symptoms': <xgboost.core.Booster at 0x780a191d81a0>,
 'Non-urgent': <xgboost.core.Booster at 0x780a191da2a0>,
 'Outpatient Logistics/Scheduling': <xgboost.core.Booster at 0x780a191da840>,
 'Physical': <xgboost.core.Booster at 0x780a191db380>,
 'Physical Environment/Climate': <xgboost.core.Booster at 0x780a191da540>,
 'Service Complaint': <xgboost.core.Booster at 0x780a191d9c40>,
 'Social Services': <xgboost.core.Booster at 0x780a191da390>,
 'Te

In [56]:
import numpy as np


In [57]:
from sklearn.metrics import f1_score


In [58]:
optimal_thresholds = {}
import numpy as np
dmatrix_val = xgb.DMatrix(x_val)

for i, label_name in enumerate(label_keep_original):

    x_val_predic = models[label_name].predict(dmatrix_val)
    best_f1 = 0
    best_thresh = 0.5
    for threshold_val in np.arange(0.05, 0.96, 0.05):
        preds = (x_val_predic > threshold_val).astype(int)
        current_f1 = f1_score(y_val[:, i], preds)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thresh = threshold_val
    optimal_thresholds[label_name] = best_thresh
    print(f"Optimal threshold for {label_name}: {best_thresh:.2f} (F1: {best_f1:.4f})")


Optimal threshold for Cultural/Religion: 0.40 (F1: 0.9485)
Optimal threshold for Diet/Nutrition: 0.45 (F1: 0.9333)
Optimal threshold for Emergent: 0.40 (F1: 0.3721)
Optimal threshold for Exercise: 0.50 (F1: 0.8571)
Optimal threshold for Friends & Family: 0.50 (F1: 0.6389)
Optimal threshold for Grateful Patient: 0.20 (F1: 0.6512)
Optimal threshold for Health Education: 0.10 (F1: 0.7333)
Optimal threshold for Laboratory/Testing: 0.30 (F1: 0.9433)
Optimal threshold for Medications: 0.45 (F1: 0.9302)
Optimal threshold for No Symptoms: 0.60 (F1: 0.8583)
Optimal threshold for Non-urgent: 0.40 (F1: 0.9512)
Optimal threshold for Outpatient Logistics/Scheduling: 0.45 (F1: 0.6970)
Optimal threshold for Physical: 0.45 (F1: 0.7485)
Optimal threshold for Physical Environment/Climate: 0.50 (F1: 0.8000)
Optimal threshold for Service Complaint: 0.20 (F1: 0.2667)
Optimal threshold for Social Services: 0.35 (F1: 0.7826)
Optimal threshold for Technical/IT: 0.55 (F1: 0.7143)
Optimal threshold for Urgent: 

In [59]:

from sklearn.metrics import f1_score
dmatrix_test = xgb.DMatrix(x_test)


for i,_ in enumerate(label_keep_original):

  x_test_predic = models[label_keep_original[i]].predict(dmatrix_test)
  y_test_final = (x_test_predic > optimal_thresholds[_]).astype(int)
  f1 = f1_score(y_test[:,i], y_test_final)
  print(f"F1 Score for {_}:", f1)


F1 Score for Cultural/Religion: 0.9345794392523364
F1 Score for Diet/Nutrition: 0.875
F1 Score for Emergent: 0.45
F1 Score for Exercise: 0.7368421052631579
F1 Score for Friends & Family: 0.6363636363636364
F1 Score for Grateful Patient: 0.5871559633027523
F1 Score for Health Education: 0.7291666666666666
F1 Score for Laboratory/Testing: 0.967741935483871
F1 Score for Medications: 0.896551724137931
F1 Score for No Symptoms: 0.8439306358381503
F1 Score for Non-urgent: 0.9514925373134329
F1 Score for Outpatient Logistics/Scheduling: 0.7611940298507462
F1 Score for Physical: 0.8095238095238095
F1 Score for Physical Environment/Climate: 0.85
F1 Score for Service Complaint: 0.06349206349206349
F1 Score for Social Services: 0.6206896551724138
F1 Score for Technical/IT: 0.5555555555555556
F1 Score for Urgent: 0.2826086956521739
F1 Score for Work/School: 0.6111111111111112


In [60]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

Y_pred_all = np.zeros(y_test.shape)

dmatrix_test = xgb.DMatrix(x_test)

print("--- Individual Class F1 Scores ---")
for i, label_name in enumerate(label_keep_original):

    x_test_predic = models[label_name].predict(dmatrix_test)

    y_test_final = (x_test_predic > optimal_thresholds[label_name]).astype(int)

    Y_pred_all[:, i] = y_test_final

    f1 = f1_score(y_test[:, i], y_test_final)
    print(f"F1 Score for {label_name}: {f1:.4f}")


macro_f1 = f1_score(y_test, Y_pred_all, average='macro', zero_division=0)
micro_f1 = f1_score(y_test, Y_pred_all, average='micro', zero_division=0)
weighted_f1 = f1_score(y_test, Y_pred_all, average='weighted', zero_division=0)

print("--- Overall Test Set Metrics ---")
print(f"Macro F1 Score:    {macro_f1:.4f}")
print(f"Micro F1 Score:    {micro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")


print("--- Full Classification Report ---")
print(classification_report(y_test, Y_pred_all, target_names=label_keep_original, zero_division=0))

--- Individual Class F1 Scores ---
F1 Score for Cultural/Religion: 0.9346
F1 Score for Diet/Nutrition: 0.8750
F1 Score for Emergent: 0.4500
F1 Score for Exercise: 0.7368
F1 Score for Friends & Family: 0.6364
F1 Score for Grateful Patient: 0.5872
F1 Score for Health Education: 0.7292
F1 Score for Laboratory/Testing: 0.9677
F1 Score for Medications: 0.8966
F1 Score for No Symptoms: 0.8439
F1 Score for Non-urgent: 0.9515
F1 Score for Outpatient Logistics/Scheduling: 0.7612
F1 Score for Physical: 0.8095
F1 Score for Physical Environment/Climate: 0.8500
F1 Score for Service Complaint: 0.0635
F1 Score for Social Services: 0.6207
F1 Score for Technical/IT: 0.5556
F1 Score for Urgent: 0.2826
F1 Score for Work/School: 0.6111
--- Overall Test Set Metrics ---
Macro F1 Score:    0.6928
Micro F1 Score:    0.8302
Weighted F1 Score: 0.8337
--- Full Classification Report ---
                                 precision    recall  f1-score   support

              Cultural/Religion       0.93      0.94  

In [61]:

import numpy as np
from sklearn.metrics import f1_score

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
n = len(y_test)

boot_scores = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    score = f1_score(y_test[idx], Y_pred_all[idx], average='macro', zero_division=0)
    boot_scores.append(score)

boot_scores = np.array(boot_scores)
lo = np.percentile(boot_scores, 2.5)
hi = np.percentile(boot_scores, 97.5)
print(f"Macro F1: {macro_f1:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Macro F1: 0.6928  95% CI [0.6609, 0.7175]


In [62]:

import numpy as np, os
os.makedirs("predictions", exist_ok=True)



np.savez_compressed(
    "predictions/XGBoost_preds.npz",
    y_true = y_test,
    y_pred = Y_pred_all,
)
print("Saved predictions/XGBoost_preds.npz")

Saved predictions/XGBoost_preds.npz


## Per-corpus + per-label evaluation (XGBoost)

Loads saved `XGBoost_preds.npz` and reports Macro F1 (+ 95% bootstrap CI) and per-label F1 separately for the pooled corpus and for the Rwanda and Canada halves. Source labels come from `Ensemble_test_predictions.json`, which carries the per-row `source_corpus` field for the same 584-doc test set.


In [ ]:
# PERCORPUSCELL_v1
# Per-corpus + per-label evaluation against the shared test split, with 1,000-resample bootstrap CIs.
import json, numpy as np
from sklearn.metrics import f1_score, classification_report

with open("predictions/Ensemble_test_predictions.json") as f:
    _ens = json.load(f)
_src    = np.array([r["source_corpus"] for r in _ens])
_labels = list(_ens[0]["true_labels"].keys())

_d = np.load("predictions/XGBoost_preds.npz")
_yt, _yp = _d["y_true"].astype(int), _d["y_pred"]
if _yp.dtype != int:
    _yp = (_yp > 0.5).astype(int) if _yp.max() <= 1 else _yp.astype(int)

for _corpus in ["Combined", "Rwanda", "Canada"]:
    _mask = np.ones(len(_src), bool) if _corpus == "Combined" else (_src == _corpus)
    _yti, _ypi = _yt[_mask], _yp[_mask]
    _macro = f1_score(_yti, _ypi, average="macro", zero_division=0)

    _rng = np.random.default_rng(42)
    _n = len(_yti)
    _boot = np.empty(1000)
    for _i in range(1000):
        _idx = _rng.integers(0, _n, _n)
        _boot[_i] = f1_score(_yti[_idx], _ypi[_idx], average="macro", zero_division=0)
    _lo, _hi = np.percentile(_boot, [2.5, 97.5])

    print(f"\n=== {{_corpus}} (n={{_n}})  Macro F1 {{_macro:.4f}}  95% CI [{{_lo:.4f}}, {{_hi:.4f}}] ===")
    print(classification_report(_yti, _ypi, target_names=_labels, zero_division=0))
